# 04 — Exploration : planchers, scorers et seuil de refus

**Projet** : Assistant documentaire RAG orchestré avec LangChain (LCEL) — chaîne prompt + retriever + citations

## Objectifs pédagogiques

1. Installer les planchers triviaux (tirage aléatoire, ordre du corpus) avant toute comparaison.
1. Comparer les scorers lexicaux à découpage, corpus et questions identiques.
1. Choisir le seuil d'abstention sur le split de calibration et lire ce que ce choix coûte en couverture.

**Objectifs transverses du dépôt**

- Découper un corpus en passages : pourquoi le chevauchement existe et comment mesurer son effet sur le rappel.
- Comprendre BM25 (saturation, normalisation de longueur, IDF) en l'implémentant, plutôt qu'en l'appelant.
- Évaluer un système de recherche par question puis moyenne : recall@k, MRR, nDCG@k et ce que chacun ignore.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (64 documents) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 64

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

CHUNKING = dict(node(CONFIG, "preprocessing").get("chunking", {}))

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (plancher contractuel : 0.6)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(
    f"Découpage         : {CHUNKING.get('max_tokens')} tokens, "
    f"chevauchement {CHUNKING.get('overlap_tokens')}"
)
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

## 1. Les planchers

Sur un corpus où chaque question désigne un thème précis, un tirage aléatoire trouve parfois un document du bon thème. Sans ce plancher, on ne sait pas si un rappel de 0,2 est un résultat ou du hasard : c'est la première ligne à écrire dans un rapport.

In [ ]:
from src.data.generators import SyntheticCorpusGenerator
from src.data.loaders import TextCorpusLoader

LOADER = TextCorpusLoader(PATHS, formats=CONFIG.data.formats)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    QUERIES = LOADER.load_queries()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} documents, {len(QUERIES)} questions")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, QUERIES, METADATA = bundle.documents, bundle.queries, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_queries(QUERIES)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} documents généré et persisté")

CALIBRATION = QUERIES[QUERIES["split"] == "calibration"].reset_index(drop=True)
VALIDATION = QUERIES[QUERIES["split"] == "val"].reset_index(drop=True)
TEST = QUERIES[QUERIES["split"] == "test"].reset_index(drop=True)
ANSWERABLE = QUERIES[QUERIES["answer_type"] != "unanswerable"].reset_index(drop=True)
# Sous-ensemble utilisé pour **régler** quoi que ce soit : la validation seule. Le test ne sert
# qu'à juger, jamais à choisir (y compris dans un notebook).
VAL_ANSWERABLE = VALIDATION[VALIDATION["answer_type"] != "unanswerable"].reset_index(drop=True)
print(
    "splits :",
    {
        name: len(frame)
        for name, frame in (("calibration", CALIBRATION), ("val", VALIDATION), ("test", TEST))
    },
)

In [ ]:
import random

from src.training.losses_metrics import retrieval_metrics

gold = [{item for item in str(ids).split(",") if item} for ids in VAL_ANSWERABLE["gold_doc_ids"]]

rng = random.Random(CONFIG.seed)
all_docs = DOCUMENTS["doc_id"].tolist()
random_ranking = [
    rng.sample(all_docs, k=min(10, len(all_docs))) for _ in range(len(VAL_ANSWERABLE))
]
corpus_order = [all_docs[:10] for _ in range(len(VAL_ANSWERABLE))]

baselines = pd.DataFrame(
    {
        "tirage aléatoire": retrieval_metrics(random_ranking, gold, ks=(1, 5, 10)),
        "ordre du corpus": retrieval_metrics(corpus_order, gold, ks=(1, 5, 10)),
    }
).T
baselines[[f"recall_at_{k}" for k in (1, 5, 10)] + ["mrr"]].round(4)

## 2. Comparer les scorers

BM25 et TF-IDF cosinus ne pondèrent pas la fréquence des termes de la même façon. Le protocole est figé — même découpage, même corpus, mêmes questions — pour que l'écart observé vienne du scorer et de rien d'autre.

In [ ]:
from src.models import available_algorithms, build_model

print("algorithmes servis par la stack :", available_algorithms())

comparison: dict[str, dict[str, float]] = {}
for algorithm in available_algorithms():
    candidate = build_model(CONFIG.model_dump(), algorithm=algorithm)
    fit_result = candidate.fit(DOCUMENTS, CALIBRATION)
    rankings = [
        [passage.doc_id for passage in candidate.retrieve(str(question), 10)]
        for question in VAL_ANSWERABLE["question"]
    ]
    metrics = retrieval_metrics(rankings, gold, ks=(1, 5, 10))
    comparison[algorithm] = {
        "passages": float(fit_result.n_chunks),
        "vocabulaire": float(fit_result.extra.get("vocabulary_size", 0.0)),
        "recall_at_1": metrics["recall_at_1"],
        "recall_at_5": metrics["recall_at_5"],
        "recall_at_10": metrics["recall_at_10"],
        "mrr": metrics["mrr"],
    }
comparison_frame = pd.DataFrame(comparison).T
comparison_frame.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison_frame[["recall_at_1", "recall_at_5", "recall_at_10"]].plot.bar(ax=axes[0], rot=0)
axes[0].set(title="Rappel par scorer (validation)", ylabel="rappel")
comparison_frame[["mrr"]].plot.bar(ax=axes[1], rot=0, color="darkorange", legend=False)
axes[1].set(title="MRR — rang du premier passage pertinent", ylabel="mrr")
plt.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Les planchers ne sont pas une formalité : ils fixent le niveau à battre, et un rapport qui les omet laisse croire qu'un score médiocre est un progrès.
- Comparer deux scorers à découpage différent ne mesure rien : le protocole doit être identique, seul le scorer change.
- Un vocabulaire plus grand n'est pas un avantage en soi : ce qui compte est de classer le bon passage en tête.

## 3. Le seuil de refus

Répondre à une question dont la réponse n'est pas dans le corpus est l'erreur la plus coûteuse d'un assistant : elle est crédible et fausse. Le seuil est réglé sur le split de **calibration**, puis on mesure ce qu'il coûte sur les questions qui, elles, ont une réponse.

In [ ]:
MODEL = build_model(CONFIG.model_dump())
FIT_RESULT = MODEL.fit(DOCUMENTS, CALIBRATION)
print(f"{FIT_RESULT.n_chunks} passages indexés en {FIT_RESULT.duration_seconds:.2f}s")
print(f"vocabulaire : {FIT_RESULT.extra.get('vocabulary_size')} termes")
print(f"seuil d'abstention retenu par la calibration : {MODEL.abstention_threshold:.4f}")

scores_answerable: list[float] = []
scores_out_of_corpus: list[float] = []
for record in CALIBRATION.to_dict(orient="records"):
    scores = [passage.score for passage in MODEL.retrieve(str(record["question"]), 5)]
    target = scores_out_of_corpus if record["answer_type"] == "unanswerable" else scores_answerable
    target.append(float(max(scores, default=0.0)))

In [ ]:
import numpy as np

candidates = np.unique(np.concatenate([scores_answerable, scores_out_of_corpus]))
sweep = pd.DataFrame(
    [
        {
            "seuil": float(candidate),
            "refus corrects": float(np.mean(np.asarray(scores_out_of_corpus) < candidate)),
            "couverture": float(np.mean(np.asarray(scores_answerable) >= candidate)),
        }
        for candidate in candidates
    ]
)
sweep["moyenne des deux"] = 0.5 * (sweep["refus corrects"] + sweep["couverture"])
best = sweep.loc[sweep["moyenne des deux"].idxmax()]
print(
    f"meilleur compromis (calibration) : seuil {best['seuil']:.3f} — "
    f"{best['refus corrects']:.0%} de refus corrects "
    f"pour {best['couverture']:.0%} de couverture"
)
display(sweep.iloc[:: max(len(sweep) // 10, 1)].round(3))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(sweep["seuil"], sweep["refus corrects"], label="refus corrects (hors corpus)")
plt.plot(sweep["seuil"], sweep["couverture"], label="couverture (questions avec réponse)")
plt.plot(
    sweep["seuil"],
    sweep["moyenne des deux"],
    linestyle="--",
    color="black",
    label="moyenne des deux",
)
plt.axvline(MODEL.abstention_threshold, color="red", linestyle=":", label="seuil retenu")
plt.xlabel("score du meilleur passage")
plt.ylabel("taux")
plt.title("Arbitrage du seuil de refus")
plt.legend()
plt.show()

**Ce qu'il faut retenir**

- Un seuil ne s'optimise pas par exactitude globale : sur un jeu majoritairement répondable, répondre toujours donne une bonne exactitude et zéro refus utile.
- L'arbitrage s'énonce en deux nombres — refus corrects et couverture — et c'est au métier de choisir le point de fonctionnement, pas au modèle.
- Si la calibration n'apporte pas un gain réel, le code conserve le seuil configuré et l'écrit dans les journaux : un réglage inventé serait pire qu'un réglage absent.